# 📘 Notebook 13: Searching

### Course: *From Zero to Data Structures & Algorithms in Python*
**Instructor:** Ioannis Tsioulis

*Module C: Algorithms & Complexity · Notebook 3 of 4 in this module · (13 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Implement **linear search** and state its cost
- Explain why **binary search** needs sorted data
- Trace binary search by hand on a small list
- Implement binary search with a loop and with recursion
- Compare the two algorithms by counting comparisons, and plot the result
- Decide which search fits a given situation

> **Prerequisites:** Notebooks 11 (Big O) and 12 (recursion).
>
> 🔭 **Why this notebook matters:** searching is the most common operation in computing. Every login, every contact you look up, every web search is a search problem. It is also the cleanest illustration of the difference between O(n) and O(log n).

## 1. Linear search

### Intuition first
You are looking for a particular exam paper in an unsorted pile. There is only one way: pick up the papers one by one until you find it, or until the pile is finished.

### Formal definition
**Linear search** examines the elements one after another, from the first to the last, and stops when it finds the target. By convention it returns the **index** of the target, or `-1` if the target is not present.

In [ ]:
def linear_search(items, target):
    for index in range(len(items)):
        if items[index] == target:
            return index          # found: stop immediately
    return -1                     # the loop ended without finding it

numbers = [34, 7, 23, 32, 5, 62, 15]

print(linear_search(numbers, 32))     # 3
print(linear_search(numbers, 100))    # -1

### How much work is it?
It depends on where the target is.

- **Best case:** the target is the first element. One comparison.
- **Worst case:** the target is the last element, or it is not there at all. `n` comparisons.
- **On average:** about `n / 2`.

In Big O terms, linear search is **O(n)**. Let us count the comparisons explicitly.

In [ ]:
def linear_search_counted(items, target):
    comparisons = 0
    for index in range(len(items)):
        comparisons += 1
        if items[index] == target:
            return index, comparisons
    return -1, comparisons

numbers = list(range(1, 1001))        # 1, 2, ..., 1000

print(linear_search_counted(numbers, 1))       # best case
print(linear_search_counted(numbers, 500))     # somewhere in the middle
print(linear_search_counted(numbers, 1000))    # worst case
print(linear_search_counted(numbers, -5))      # not present: also the worst case

> 🧠 Linear search has one great virtue: it works on **any** list, in any order. It asks nothing of the data. Python's own `x in my_list` and `my_list.index(x)` are linear searches.

## 2. Binary search

### Intuition first
Now you are looking for a word in a printed dictionary. You would never start at page 1. You open the book near the middle, look at the word there, and immediately know which **half** your word is in. You discard the other half and repeat. Each look halves what is left.

That only works because the dictionary is **sorted**. If the words were in random order, looking at the middle would tell you nothing.

### Formal definition
**Binary search** works on a **sorted** list. It keeps two markers, `low` and `high`, enclosing the part of the list where the target may still be.

1. Look at the element in the middle of that part.
2. If it is the target, we are done.
3. If the middle element is **smaller** than the target, the target can only be to the **right**: move `low` just past the middle.
4. If it is **larger**, the target can only be to the **left**: move `high` just before the middle.
5. Repeat until the target is found, or until `low` passes `high`, which means there is nowhere left to look.

### Tracing it by hand
Let us search for **23** in a sorted list of 10 numbers.

```
index:   0   1   2   3   4   5   6   7   8   9
value:   2   5   8  12  16  23  38  56  72  91

Step 1:  low=0, high=9, mid=4 -> value 16.  16 < 23, so look right: low=5
Step 2:  low=5, high=9, mid=7 -> value 56.  56 > 23, so look left:  high=6
Step 3:  low=5, high=6, mid=5 -> value 23.  Found at index 5.
```
Three comparisons instead of six.

In [ ]:
def binary_search(items, target):
    low = 0
    high = len(items) - 1

    while low <= high:
        mid = (low + high) // 2
        if items[mid] == target:
            return mid
        elif items[mid] < target:
            low = mid + 1          # discard the left half
        else:
            high = mid - 1         # discard the right half
    return -1

numbers = [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]

print(binary_search(numbers, 23))     # 5
print(binary_search(numbers, 2))      # 0
print(binary_search(numbers, 50))     # -1

### Watching it work
The version below prints the part of the list still under consideration at each step.

In [ ]:
def binary_search_traced(items, target):
    low = 0
    high = len(items) - 1
    step = 1

    while low <= high:
        mid = (low + high) // 2
        print(f"Step {step}: searching {items[low:high + 1]}, middle value {items[mid]}")
        if items[mid] == target:
            print(f"Found {target} at index {mid}.")
            return mid
        elif items[mid] < target:
            low = mid + 1
        else:
            high = mid - 1
        step += 1

    print(f"{target} is not in the list.")
    return -1

binary_search_traced(numbers, 72)
print()
binary_search_traced(numbers, 13)

> ⚠️ **Common pitfalls**
>
> - Using binary search on an **unsorted** list. It does not crash. It silently returns wrong answers, which is worse.
> - Writing `low = mid` instead of `low = mid + 1`. The middle element has already been checked. Keeping it can make the loop run forever.
> - Writing `while low < high`. With `<` the last remaining element is never examined. The condition must be `<=`.

## 3. How fast is binary search?

Each comparison halves the remaining part. Starting from `n` elements we have `n`, `n/2`, `n/4`, ... until one element is left. The number of halvings is the logarithm of `n`, so binary search is **O(log n)**.

Let us count the comparisons for the worst case of both algorithms, on lists of very different sizes.

In [ ]:
def binary_search_counted(items, target):
    low = 0
    high = len(items) - 1
    comparisons = 0

    while low <= high:
        comparisons += 1
        mid = (low + high) // 2
        if items[mid] == target:
            return mid, comparisons
        elif items[mid] < target:
            low = mid + 1
        else:
            high = mid - 1
    return -1, comparisons

print(f"{'n':>12} {'linear':>12} {'binary':>8}")
for n in [10, 100, 1_000, 100_000, 1_000_000]:
    numbers = list(range(n))
    target = -1                                   # not present: the worst case
    linear = linear_search_counted(numbers, target)[1]
    binary = binary_search_counted(numbers, target)[1]
    print(f"{n:>12,} {linear:>12,} {binary:>8}")

A list of a million elements needs a million comparisons with linear search and **twenty** with binary search. Multiply the size of the list by a thousand, and binary search needs only ten more steps.

In [ ]:
import matplotlib.pyplot as plt

sizes = list(range(1, 1001, 10))
linear_counts = []
binary_counts = []

for n in sizes:
    numbers = list(range(n))
    linear_counts.append(linear_search_counted(numbers, -1)[1])
    binary_counts.append(binary_search_counted(numbers, -1)[1])

plt.figure(figsize=(7, 4))
plt.plot(sizes, linear_counts, label="Linear search: O(n)")
plt.plot(sizes, binary_counts, label="Binary search: O(log n)")
plt.title("Comparisons in the worst case")
plt.xlabel("Size of the list (n)")
plt.ylabel("Comparisons")
plt.legend()
plt.grid(True)
plt.show()

### And on the clock
Counting is clean. For completeness, here is the same comparison in real time, with 500 searches in a list of 100,000 elements.

In [ ]:
import time
import random

numbers = list(range(100_000))
targets = [random.randint(0, 100_000) for i in range(500)]

start = time.perf_counter()
for target in targets:
    linear_search(numbers, target)
linear_time = time.perf_counter() - start

start = time.perf_counter()
for target in targets:
    binary_search(numbers, target)
binary_time = time.perf_counter() - start

print(f"Linear search: {linear_time:.4f} seconds")
print(f"Binary search: {binary_time:.4f} seconds")
print(f"Binary search was about {linear_time / binary_time:,.0f} times faster.")

## 4. Binary search, recursively

Binary search fits recursion naturally: *look at the middle; if it is not the target, the problem is the same search on one half.* The base case is an empty range.

In [ ]:
def binary_search_recursive(items, target, low, high):
    if low > high:                 # base case: nothing left to search
        return -1
    mid = (low + high) // 2
    if items[mid] == target:
        return mid
    elif items[mid] < target:
        return binary_search_recursive(items, target, mid + 1, high)
    else:
        return binary_search_recursive(items, target, low, mid - 1)

numbers = [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]
print(binary_search_recursive(numbers, 56, 0, len(numbers) - 1))
print(binary_search_recursive(numbers, 3, 0, len(numbers) - 1))

## 5. Which search should I use?

Binary search is dramatically faster, but it has a price: **the data must be sorted, and it must stay sorted**. Sorting costs O(n log n), as we will see in the next notebook. That is more than a single linear search.

| Situation | Best choice |
|---|---|
| a small list, or a single search | linear search |
| unsorted data that changes all the time | linear search |
| a large list that is searched **many times** | sort it once, then binary search |
| you only need to know *whether* a value exists | a **set** or a **dictionary**, O(1) |

The last row deserves a comment. In Notebook 11 we saw that a set answers "is this value present?" in constant time, faster even than binary search. Why then learn binary search at all? Because sorted data can answer questions a set cannot: *what is the nearest value? how many values lie between 10 and 20? what is the next larger value?* The exercises touch on some of these.

> 🧠 Python has binary search built in, in the `bisect` module. `bisect.bisect_left(sorted_list, x)` returns the position where `x` is, or where it would be inserted. In real projects use it. In this course, write your own first, so that you know what it does.

---
## ✏️ Exercises

### Exercise 1 (Search for a name)
Use a linear search to find the index of `"Eleni"` in the list below. Then search for `"Zoe"`.

In [ ]:
names = ["Anna", "Kostas", "Eleni", "Dimitris", "Maria"]
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
names = ["Anna", "Kostas", "Eleni", "Dimitris", "Maria"]

def linear_search(items, target):
    for index in range(len(items)):
        if items[index] == target:
            return index
    return -1

print(linear_search(names, "Eleni"))
print(linear_search(names, "Zoe"))
```
</details>

### Exercise 2 (Find all positions)
Write a function `find_all(items, target)` that returns a **list of all the indices** where `target` appears. For `[4, 2, 4, 7, 4]` and target `4` the answer is `[0, 2, 4]`.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def find_all(items, target):
    positions = []
    for index in range(len(items)):
        if items[index] == target:
            positions.append(index)
    return positions

print(find_all([4, 2, 4, 7, 4], 4))
```

*This search cannot stop early: it must always look at every element, so it is O(n) even in the best case.*
</details>

### Exercise 3 (Search in a list of records)
Write a function `find_student(students, name)` that returns the **dictionary** of the student with the given name, or `None` if there is no such student.

In [ ]:
students = [
    {"name": "Anna", "grade": 8},
    {"name": "Kostas", "grade": 6},
    {"name": "Eleni", "grade": 9},
]
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
students = [
    {"name": "Anna", "grade": 8},
    {"name": "Kostas", "grade": 6},
    {"name": "Eleni", "grade": 9},
]

def find_student(students, name):
    for student in students:
        if student["name"] == name:
            return student
    return None

print(find_student(students, "Kostas"))
print(find_student(students, "Zoe"))
```
</details>

### Exercise 4 (Trace by hand)
On paper, trace binary search for the target `7` in `[1, 3, 5, 7, 9, 11, 13, 15, 17]`. Write down `low`, `high` and `mid` at each step. Then check your trace with `binary_search_traced`.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
# Step 1: low=0, high=8, mid=4 -> value 9.  9 > 7, so high = 3
# Step 2: low=0, high=3, mid=1 -> value 3.  3 < 7, so low = 2
# Step 3: low=2, high=3, mid=2 -> value 5.  5 < 7, so low = 3
# Step 4: low=3, high=3, mid=3 -> value 7.  Found at index 3.
binary_search_traced([1, 3, 5, 7, 9, 11, 13, 15, 17], 7)
```
</details>

### Exercise 5 (Binary search on words)
Binary search works on anything that can be compared, including text. Search for `"mango"` and for `"grape"` in the sorted list below, using `binary_search`.

In [ ]:
fruits = ["apple", "banana", "cherry", "kiwi", "lemon", "mango", "orange", "pear"]
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
fruits = ["apple", "banana", "cherry", "kiwi", "lemon", "mango", "orange", "pear"]
print(binary_search(fruits, "mango"))
print(binary_search(fruits, "grape"))
```
</details>

### Exercise 6 (How many guesses?)
Write a function `max_steps(n)` that returns the largest number of comparisons binary search can need on a list of `n` elements, by repeatedly halving `n`. Print the result for 100, 1,000, 1,000,000 and 1,000,000,000.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def max_steps(n):
    steps = 0
    while n > 0:
        n = n // 2
        steps += 1
    return steps

for n in [100, 1_000, 1_000_000, 1_000_000_000]:
    print(f"{n:>13,}: {max_steps(n)} comparisons")
```

*A thousand million elements, thirty comparisons. This is why every large database keeps its data in sorted or tree-shaped structures.*
</details>

### Exercise 7 (Where would it go?)
Write a function `insert_position(items, target)` for a sorted list. It returns the index of `target` if it is present, and otherwise the index where it **should be inserted** to keep the list sorted. For `[10, 20, 30, 40]`, the target `25` gives `2` and the target `5` gives `0`. (Hint: run an ordinary binary search. When the loop ends without finding the target, look at where `low` is.)

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def insert_position(items, target):
    low = 0
    high = len(items) - 1
    while low <= high:
        mid = (low + high) // 2
        if items[mid] == target:
            return mid
        elif items[mid] < target:
            low = mid + 1
        else:
            high = mid - 1
    return low

print(insert_position([10, 20, 30, 40], 25))
print(insert_position([10, 20, 30, 40], 5))
print(insert_position([10, 20, 30, 40], 99))
```

*When the search fails, `low` has stopped exactly at the first element larger than the target. This is a question a set cannot answer.*
</details>

### Exercise 8 (Integer square root, a little harder)
Binary search does not need a list. It needs only a **sorted range of possible answers**. Write `integer_sqrt(n)` that returns the largest whole number whose square is not greater than `n`, by binary search over the numbers 0 to `n`. For example `integer_sqrt(50)` is `7`. Do not use `**0.5` or the `math` module.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def integer_sqrt(n):
    low = 0
    high = n
    answer = 0
    while low <= high:
        mid = (low + high) // 2
        if mid * mid <= n:
            answer = mid           # mid is a candidate; try something larger
            low = mid + 1
        else:
            high = mid - 1         # too big: look lower
    return answer

print(integer_sqrt(50))
print(integer_sqrt(144))
print(integer_sqrt(10 ** 12))
```

*The last call searches a range of a million million numbers in about 40 steps. Searching over a range of answers instead of a list is a powerful and widely used technique.*
</details>

## 🔑 Key takeaways

- **Linear search** checks the elements one by one: **O(n)**, and it works on any list.
- **Binary search** repeatedly halves a **sorted** list: **O(log n)**.
- A million elements: up to a million comparisons for linear search, about twenty for binary search.
- Binary search on unsorted data gives wrong answers without any warning.
- The details matter: `while low <= high`, `low = mid + 1`, `high = mid - 1`.
- Sorting has a cost, so binary search pays off when the same data is searched **many times**.

---
**Next:** *Notebook 14: Sorting*, because binary search needs sorted data, and somebody has to sort it.

---
*© Ioannis Tsioulis. *From Zero to Data Structures & Algorithms in Python*. Prepared for educational use.*